# Learned arms served live from a portable document: the same prediction in Python and in the browser

**Level 4: Beyond the state of the art** · Needs: 05-learned-arms/01-fitting-a-learned-arm-on-nine-sites, 06-design-response/02-a-burden-and-spacing-surface · Reading time: about 30 minutes

## Objectives

The web application recomputes every model, the learned ones included, on a design the reader changes, with no Python server. That works because each learned arm is exported once to a portable JSON document that a browser can evaluate (claim BS-5). This notebook opens one such document, evaluates it with the engine's reference reader, and shows what makes it trustworthy.

By the end you will have:

- opened the models file of the Enusa scope and listed its seven arms, their kinds and their size on disk;
- checked its content address, then shown that a one-number tamper changes it;
- evaluated the fixtures it carries, compared the portable prediction with a fresh fit of the engine on the same training rows, and moved a design lever to see both follow.

## Setup

Install the packages with the exact pins of `notebooks/requirements.txt`. The engine is `blastfrag`, published on PyPI; the other packages are its numerical base and the plotting library.

```bash
pip install blastfrag==0.4.0 numpy==2.5.3 matplotlib==3.11.2
```

This cell imports nothing from the notebook yet: it reads the installed versions from the package metadata and prints them, so the output records exactly which engine produced every number below.

In [1]:
# Print the installed version of every package this notebook uses
from importlib.metadata import version

for name in ("blastfrag", "numpy", "matplotlib",):
    print(f"{name}=={version(name)}")

blastfrag==0.4.0
numpy==2.5.3
matplotlib==3.11.2


## Data

| Source | Owner and URL | Licence class | File the notebook reads |
|---|---|---|---|
| The training rows | Hudaverdi, Kulatilake and Kuzu 2010, https://doi.org/10.1002/nag.957 | numeric facts reused with citation; the article never ships | the corpus inside the `blastfrag` wheel |
| The models file of the Enusa scope: the seven arms fitted without any Enusa blast, with their fixtures | this repository (MIT), written by the offline bake | the product's own licence | `data/derived/models/Enusa.json` |
| The index, which lists each models file with its digest | this repository (MIT), written by the offline bake | the product's own licence | `data/derived/manifests/index.json` |

The reader of the portable format is `bf.predict_portable`, in the engine. The browser has its own reader in TypeScript; the product's test `frontend/test/parity.test.ts` checks it against the committed documents.

## Method

A fitted learned arm is not a pickle or a model server. The engine writes each one as a JSON document in a versioned schema (`blastfrag.portable/v1`) that holds exactly what a prediction needs: the trees as arrays of left and right children, split features and thresholds for the forest and the boosted model; the standardisation constants; the support vectors and dual coefficients for the support vector regressor; the layer weights for the network; the coefficients for the refitted power law; and, for the stacking ensemble, a reference to the forest and the boosted model plus the meta-learner. Reading it needs arithmetic only, so a reader written in TypeScript can reproduce it in a browser.

A models file bundles the seven arms of one training scope, here the nine sites that remain when Enusa is held out, and ends with a content address, the SHA-256 of the file's canonical serialisation. It also carries fixtures: input vectors and, per arm, the full-precision prediction of the original fitted model, which any reader of the format can be tested against.

## State of the art

- The learned predictors published for this corpus are reported through their scores and their parameters: the network widths (Kulatilake et al. 2012, https://doi.org/10.1007/s10706-012-9496-3), the support vector kernels and constants (Amoako et al. 2022, https://doi.org/10.3390/mining2020013; Sui et al. 2025, https://doi.org/10.3390/app15031254) and the boosting learning rate (Sui et al. 2025).
- The sources the research held record no runnable form of any of them that a reader could apply to a new design. A reader who wants the prediction for a design has to retrain the model from the parameters.
- The conventional way to serve a learned model is a Python process behind an API, which the product's static deployment does not have.

## Beyond the state of the art

### Step 1: open the models file and check its content address

The cell loads the Enusa models file and the index, lists the arms with their kind, and recomputes the content address with the pipeline's own `digest` function, which hashes the canonical serialisation of the payload without its `digest` field. It compares the result with the digest stored in the file and with the one the index lists.

In [2]:
# Imports, the models file of the Enusa scope, its arms and its content address
import copy
import json
import sys
from pathlib import Path

import blastfrag as bf
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path("../../../data-pipeline")))
from pipeline.cases.fragmenta_cases import Variant
from pipeline.model.blasts import apply_variant
from pipeline.stages.export import digest

DERIVED = Path("../../../data/derived")
path = DERIVED / "models" / "Enusa.json"
document = json.loads(path.read_text(encoding="utf-8"))
index = json.loads((DERIVED / "manifests" / "index.json").read_text(encoding="utf-8"))
print("scope:", document["scope"], "| held-out site:", document["held_out_site"], "| training rows:", document["n_training_rows"], "| schema:", document["schema"], "/", document["portable_schema"])
print("file size:", path.stat().st_size, "bytes")
for name, arm in document["arms"].items():
    print(f"  {name:<22}{arm['kind']:<20}{len(json.dumps(arm)):>9} characters of JSON")

payload = {k: v for k, v in document.items() if k != "digest"}
recomputed = digest(payload)
listed = next(m["digest"] for m in index["models"] if m["scope"] == "Enusa")
print("digest recomputed equals the stored one:", recomputed == document["digest"], "| equals the index:", recomputed == listed)

scope: Enusa | held-out site: Enusa | training rows: 85 | schema: fragmenta.models/v1 / blastfrag.portable/v1
file size: 238371 bytes
  kuznetsov-transfer    kuznetsov-transfer        891 characters of JSON
  published-neural-net  network                 24782 characters of JSON
  random-forest         forest                 168687 characters of JSON
  refitted-regression   power-law                 992 characters of JSON
  stacking              stacking                  890 characters of JSON
  svr-rbf               svr                      9114 characters of JSON
  xgboost               xgboost                 40032 characters of JSON
digest recomputed equals the stored one: True | equals the index: True


The next cell tampers with a copy: it changes one leaf value of the random forest by a hundredth of a millimetre and recomputes the address. A content address that did not move would be useless.

In [3]:
# Change one number of a copy and recompute the content address
tampered = copy.deepcopy(payload)
tampered["arms"]["random-forest"]["trees"][0]["t"][-1] += 1e-5
print("address of the tampered copy differs:", digest(tampered) != document["digest"])

address of the tampered copy differs: True


### Step 2: the fixtures and a fresh fit

A reader of the format is correct if it reproduces the fixtures. The cell evaluates every fixture input with `predict_portable` and compares with the expected values stored at full precision. The stacking document holds references to the forest and the boosted model, so the cell resolves them first. For the transfer arm the value is the rock factor, not a size.

It then goes one step further: it fits the engine's arms again on the same training rows (every blast outside Enusa) and compares the portable prediction with the fresh arm on all 97 blasts.

In [4]:
# Portable against stored fixtures, and portable against a fresh fit of the engine on the same rows
arms = document["arms"]

def readable(name):
    # A stacking document refers to the forest and the boosted model, so put them in before reading
    doc = copy.deepcopy(arms[name])
    if doc["kind"] == "stacking":
        doc["forest"] = arms[doc["forest"]["ref"]]
        doc["boosting"] = arms[doc["boosting"]["ref"]]
    return doc

fixtures = document["fixtures"]
print("fixture inputs:", len(fixtures["inputs"]))
print(f"{'arm':<22}{'largest difference to the fixtures':>44}")
for name in arms:
    gaps = []
    for item, want in zip(fixtures["inputs"], fixtures["expected"][name]):
        value, _ = bf.predict_portable(readable(name), item["features"])
        if value is not None and want is not None:
            gaps.append(abs(value - want))
    print(f"{name:<22}{max(gaps):>44.2e}")

train = bf.load_training_corpus()
fit_rows = [b for b in train if b.site != document["held_out_site"]]
print("fresh fit on", len(fit_rows), "rows, compared on all", len(train), "blasts")
factories = bf.default_arms()
for name in [n for n in arms if n != "kuznetsov-transfer"]:
    fresh = factories[name]().fit(fit_rows)
    gaps = []
    for blast in train:
        engine = fresh.predict_one(blast).x50_m
        portable, _ = bf.predict_portable(readable(name), blast)
        if engine is not None and portable is not None:
            gaps.append(abs(engine - portable))
    print(f"{name:<22} largest difference engine against portable = {max(gaps):.2e} m  ({len(gaps)} blasts)")

fixture inputs: 116
arm                             largest difference to the fixtures
kuznetsov-transfer                                        0.00e+00
published-neural-net                                      8.88e-16
random-forest                                             0.00e+00
refitted-regression                                       0.00e+00
stacking                                                  0.00e+00
svr-rbf                                                   2.78e-15
xgboost                                                   0.00e+00
fresh fit on 85 rows, compared on all 97 blasts
published-neural-net   largest difference engine against portable = 8.88e-16 m  (97 blasts)
random-forest          largest difference engine against portable = 0.00e+00 m  (97 blasts)
refitted-regression    largest difference engine against portable = 0.00e+00 m  (97 blasts)
stacking               largest difference engine against portable = 0.00e+00 m  (97 blasts)
svr-rbf                large

### Step 3: move a design lever

The point of a portable document is that the prediction follows a design the reader changes. The cell applies the product's `burden-tight` variant (burden 15 percent tighter) to an Enusa blast and predicts the base and the perturbed design with the portable document and with the fresh engine arm, for the gradient-boosting and the stacking arms.

In [5]:
# The base and a tighter-burden design, predicted from the portable document and from the engine
blast = next(b for b in train if b.blast_id == "En1")
tight = apply_variant(blast, Variant("burden-tight", "", "", "B_over_D", 0.85))
print(f"{'arm':<12}{'design':<22}{'portable (m)':>16}{'engine (m)':>14}")
for name in ("xgboost", "stacking"):
    fresh = factories[name]().fit(fit_rows)
    for label, design in (("base", blast), ("burden-tight", tight)):
        portable, _ = bf.predict_portable(readable(name), design)
        print(f"{name:<12}{label:<22}{portable:>16.4f}{fresh.predict_one(design).x50_m:>14.4f}")

arm         design                    portable (m)    engine (m)
xgboost     base                            0.4601        0.4601
xgboost     burden-tight                    0.4600        0.4600
stacking    base                            0.4606        0.4606
stacking    burden-tight                    0.4604        0.4604


## Results

**The document is a complete, self-contained model.** The seven arms of the Enusa scope occupy a few hundred kilobytes of JSON and need arithmetic only. The reference reader reproduces every fixture to float precision.

**It is the same model as the engine's.** A fresh fit of the engine on the same 85 training rows gives the same predictions as the portable document on all 97 blasts, to floating-point rounding (differences of the order of 1e-15 m). The browser therefore recomputes the model that was fitted and scored, not a re-implementation of it that could drift.

**It is tamper-evident.** The content address recomputed with the pipeline's function equals the stored one and the index's, and a change of one number in one leaf moves it. A stale or edited document cannot pass as the committed one.

**It follows a design.** Tightening the burden moves the portable and the engine prediction together; for the tree-based arms the move is small and may be zero, as the staircase of the previous notebook showed.

**What this does not show.** Parity of two readers says the browser computes the same function, not that the function is accurate: the learned arms still score near or below zero held out by site. And the transfer arm's document returns a rock factor, not a size, so a reader must apply the classical equation to it.

## Summary

- Each learned arm is exported once to a portable JSON document; the browser evaluates it with arithmetic only (claim BS-5).
- The reference reader reproduces the stored fixtures, and a fresh fit of the engine on the same rows gives the same predictions.
- A content address makes the document tamper-evident, and the prediction follows a design lever.
- This closes the set: the data, the classical equation, the curves, the protocol, the learned arms and the live response.

## References

- Kulatilake, Hudaverdi and Wu 2012, https://doi.org/10.1007/s10706-012-9496-3 (the network)
- Amoako, Jha and Zhong 2022, https://doi.org/10.3390/mining2020013 (support vector regression)
- Sui, Zhou, Zhao, Yang and Zou 2025, https://doi.org/10.3390/app15031254 (forest, boosting, stacking)
- Hudaverdi, Kulatilake and Kuzu 2010, https://doi.org/10.1002/nag.957 (the corpus)
- `blastfrag` 0.4.0, the engine, on PyPI: https://pypi.org/project/blastfrag/